In [20]:
import pandas as pd
import numpy as np

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from feature_engine.outliers import Winsorizer

from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

import joblib
import warnings
warnings.filterwarnings("ignore")

In [21]:
dataset = pd.read_csv("Airline_Passenger_Satisfaction.csv")
dataset.head()

,satisfaction,Customer Type,Age,Type of Travel,Class,Flight Distance,Seat comfort,Departure/Arrival time convenient,Food and drink,Gate location,...,Online support,Ease of Online booking,On-board service,Leg room service,Baggage handling,Checkin service,Cleanliness,Online boarding,Departure Delay in Minutes,Arrival Delay in Minutes
0,neutral or dissatisfied,Loyal Customer,39,Business travel,Business,4983,5,4,4,3,...,5,5,4,5,5,3,4,5,0,9
1,satisfied,Loyal Customer,60,Business travel,Business,4983,4,4,5,2,...,5,4,4,5,4,5,4,5,0,13
2,satisfied,Loyal Customer,37,Business travel,Business,2200,5,3,3,3,...,5,4,4,5,5,4,4,5,4,29
3,neutral or dissatisfied,Disloyal Customer,40,Business travel,Business,1004,5,3,4,3,...,3,4,2,5,2,3,4,3,14,23
4,neutral or dissatisfied,Loyal Customer,39,Business travel,Eco,1663,2,3,4,2,...,4,4,1,2,3,3,3,3,58,74


In [22]:
x = dataset.drop(["satisfaction"], axis=1)
y = dataset["satisfaction"]

In [23]:
x.head()

,Customer Type,Age,Type of Travel,Class,Flight Distance,Seat comfort,Departure/Arrival time convenient,Food and drink,Gate location,Inflight wifi service,...,Online support,Ease of Online booking,On-board service,Leg room service,Baggage handling,Checkin service,Cleanliness,Online boarding,Departure Delay in Minutes,Arrival Delay in Minutes
0,Loyal Customer,39,Business travel,Business,4983,5,4,4,3,3,...,5,5,4,5,5,3,4,5,0,9
1,Loyal Customer,60,Business travel,Business,4983,4,4,5,2,2,...,5,4,4,5,4,5,4,5,0,13
2,Loyal Customer,37,Business travel,Business,2200,5,3,3,3,4,...,5,4,4,5,5,4,4,5,4,29
3,Disloyal Customer,40,Business travel,Business,1004,5,3,4,3,2,...,3,4,2,5,2,3,4,3,14,23
4,Loyal Customer,39,Business travel,Eco,1663,2,3,4,2,2,...,4,4,1,2,3,3,3,3,58,74


In [24]:
y.head()

0    neutral or dissatisfied
1                  satisfied
2                  satisfied
3    neutral or dissatisfied
4    neutral or dissatisfied
Name: satisfaction, dtype: object

In [25]:
numeric_feature = x.select_dtypes(exclude=["object"]).columns
catag_feature = x.select_dtypes(include=["object"]).columns

In [26]:
numeric_feature

Index(['Age', 'Flight Distance', 'Seat comfort',
       'Departure/Arrival time convenient', 'Food and drink', 'Gate location',
       'Inflight wifi service', 'Inflight entertainment', 'Online support',
       'Ease of Online booking', 'On-board service', 'Leg room service',
       'Baggage handling', 'Checkin service', 'Cleanliness', 'Online boarding',
       'Departure Delay in Minutes', 'Arrival Delay in Minutes'],
      dtype='object')

In [27]:
catag_feature

Index(['Customer Type', 'Type of Travel', 'Class'], dtype='object')

In [28]:
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("winsorizer", Winsorizer(capping_method="iqr", tail="both", fold=1.5)),
    ("scaler", MinMaxScaler())
])

In [29]:
catag_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(drop="first", handle_unknown="ignore", sparse_output=False))
])

In [30]:
preprocess_pipeline = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_feature),
    ("catag", catag_pipeline, catag_feature)
])

In [31]:
clean = preprocess_pipeline.fit(x)

In [32]:
Clean_data = pd.DataFrame(clean.transform(x))

In [33]:
Clean_data.head()

,0,1,2,3,4,5,6,7,8,9,...,12,13,14,15,16,17,18,19,20,21
0,0.470588,1.000000,1.000000,0.714286,0.8,0.428571,0.6,0.428571,1.000000,1.000000,...,1.000000,0.428571,0.714286,1.000000,0.000000,0.189474,1.0,0.0,0.0,0.0
1,0.779412,1.000000,0.714286,0.714286,1.0,0.142857,0.4,0.714286,1.000000,0.714286,...,0.714286,1.000000,0.714286,1.000000,0.000000,0.273684,1.0,0.0,0.0,0.0
2,0.441176,0.440869,1.000000,0.428571,0.6,0.428571,0.8,0.714286,1.000000,0.714286,...,1.000000,0.714286,0.714286,1.000000,0.123077,0.610526,1.0,0.0,0.0,0.0
3,0.485294,0.190974,1.000000,0.428571,0.8,0.428571,0.4,0.142857,0.428571,0.714286,...,0.142857,0.428571,0.714286,0.428571,0.430769,0.484211,0.0,0.0,0.0,0.0
4,0.470588,0.328667,0.142857,0.428571,0.8,0.142857,0.4,0.142857,0.714286,0.714286,...,0.428571,0.428571,0.428571,0.428571,1.000000,1.000000,1.0,0.0,1.0,0.0


In [34]:
joblib.dump(clean, "naive_bayes_preprocess_pipeline.pkl")

['naive_bayes_preprocess_pipeline.pkl']

In [35]:
joblib.load("naive_bayes_preprocess_pipeline.pkl")

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('numeric', ...), ('catag', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``feature_name``. 

In [36]:
x_train, x_test, y_train, y_test = train_test_split(
    Clean_data, y, test_size=0.2, random_state=42
)

In [38]:
model = MultinomialNB()

In [39]:
model.fit(x_train, y_train)

,"alpha alpha: float or array-like of shape (n_features,), default=1.0Additive (Laplace/Lidstone) smoothing parameter(set alpha=0 and force_alpha=True, for no smoothing).",1.0
,"force_alpha force_alpha: bool, default=TrueIf False and alpha is less than 1e-10, it will set alpha to1e-10. If True, alpha will remain unchanged. This may causenumerical errors if alpha is too close to 0... versionadded:: 1.2.. versionchanged:: 1.4 The default value of `force_alpha` changed to `True`.",True
,"fit_prior fit_prior: bool, default=TrueWhether to learn class prior probabilities or not.If false, a uniform prior will be used.",True
,"class_prior class_prior: array-like of shape (n_classes,), default=NonePrior probabilities of the classes. If specified, the priors are notadjusted according to the data.",None


In [40]:
y_train_pred = model.predict(x_train)
y_test_pred = model.predict(x_test)

In [41]:
print("Training Accuracy:", accuracy_score(y_train, y_train_pred))
print("Testing Accuracy:", accuracy_score(y_test, y_test_pred))

Training Accuracy: 0.6985390360332615
Testing Accuracy: 0.6977209732060363


In [42]:
print("\nClassification Report:")
print(classification_report(y_test, y_test_pred))


Classification Report:
                         precision    recall  f1-score   support

neutral or dissatisfied       0.63      0.47      0.54      9799
              satisfied       0.72      0.83      0.77     16177

               accuracy                           0.70     25976
              macro avg       0.68      0.65      0.66     25976
           weighted avg       0.69      0.70      0.69     25976



In [43]:
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_test_pred))


Confusion Matrix:
[[ 4641  5158]
 [ 2694 13483]]


In [45]:
joblib.dump(model, "naive_bayes.pkl")

['naive_bayes.pkl']